# SAEM in practice: the Mavoglurant use-case

This notebook demonstrates stochastic approximation expectation-maximization (SAEM) [1, 2] 
using the Mavoglurant use-case [3]. 

## 1. Prepare the environment

Import the required Python dependencies

In [ ]:
from vpop_calibration.api import *
import pandas as pd
from plotnine import *
import numpy as np

# Seed for the random number generator for reproducibility: the same seed always gives the same results
# Pick your own seed (any integer) such that we all get different results
YOUR_SEED = 42

## 2. Load and prepare the Mavoglurant observations

### About the dataset

Mavoglurant is a drug investigated for the treatment of [fragile X syndrome](https://en.wikipedia.org/wiki/Fragile_X_syndrome).

The dataset is the same as in nlmixr2's R vignette [3].
It pools **120 participants** who each received a single intravenous infusion of mavoglurant at one of three doses: 25, 37.5 or 50 mg.  
It contains longitudinal data of the **venous plasma concentration of mavoglurant** (`DV` which we will compare with the model output `C15`) as well as some patients covariates (age, weight, etc.)

In [ ]:
# If for some reason the files are not there, uncomment these two lines
# !wget -q -N https://raw.githubusercontent.com/novainsilico/vpop-calibration/refs/heads/main/examples/benchmarking/Mavoglurant/CM_Mavoglurant.xml
# !wget -q -N https://raw.githubusercontent.com/novainsilico/vpop-calibration/refs/heads/main/examples/benchmarking/Mavoglurant/Mavoglurant_Dataset.csv

df = pd.read_csv("./Mavoglurant_Dataset.csv")
obs_df = (
    df.loc[df["EVID"] == 0]
    .rename(columns={"ID": "id", "TIME": "time", "DV": "value", "DOSE": "Dose"})[
        ["id", "time", "value", "Dose", "WT"]
    ]
    .astype({"value": "float"})
)
# convert from hours to seconds
obs_df["time"] = obs_df["time"].apply(lambda t: t * 60 * 60)
# use the log of the plasma concentration: log(C15)
obs_df["value"] = obs_df["value"].apply(lambda t: np.log(t))
obs_df["output_name"] = "logC15"
obs_df["protocol_arm"] = "identity"
(
    ggplot(obs_df, aes(x="time / 3600", y="value", group="id"))
    + geom_line(alpha=0.6)
    + geom_point(size=1, alpha=0.8)
    + theme(legend_position="none")
    + labs(title="Mavoglurant 120-patient dataset", x="time (h)", y="log(C15)")
)

## 3. Objectives of the SAEM calibration

We want to answer two questions from these 120 profiles:

1. **What is the typical pharmacokinetics of mavoglurant**, expressed through physiological PBPK parameters (hepatic clearance, tissue partitioning)?
2. **How much do participants differ**, and how much of the spread seen above is true between-participant variability rather than residual noise?

Each participant has only 11–15 plasma observations, too few to reliably identify 6 PBPK parameters one participant at a time. A [**nonlinear mixed-effects (NLME)** model](../../docs/nlme_model.md) instead estimates a population distribution of parameters and lets every participant "borrow strength" from the others. SAEM estimates this model by maximizing the observed likelihood [1, 2] (more details on the implementation [here](../../docs/saem_implementation.md)).

![SAEM calibration workflow](./saem_workflow.png)

By the end of the notebook we will have:

  * population estimates: typical values $\beta$, variability $\Omega$, residual error $\sigma^2$ (7.)
  * evidence that the fitted population reproduces the study: PWRES, NPDE, VPC and a log-likelihood (8.)
  * individual parameter estimates, their shrinkage, and a log-likelihood (9.)
  * a virtual population ready for mechanistic simulation (10.)


## 4. The structural model

We use a 16-compartment whole-body PBPK model for mavoglurant [4] 

![PBPK model](./mavoglurant_pbpk.png)


The model is directly loaded from an SBML document. 
The simulations of the resulting ODE system will be carried out by `libRoadRunner`. 
For a proposed set of individual parameters, it solves the model at the observation times and returns `logC15`.

The eight named inputs connect the statistical model to the simulator: six parameters will be estimated with interindividual variability, while `Dose` and `WT` (the patient weight) come directly from the data.

In [ ]:
struct_model = StructuralSbml(
    model_path="./CM_Mavoglurant.xml",
    inputs=["KbBR", "CLint", "KbMU", "KbAD", "KbBO", "KbRB", "Dose", "WT"],
    outputs=["logC15"],
)

# ODE solver settings
struct_model.rr.setIntegrator("cvode")
integ = struct_model.rr.integrator
integ.setValue("stiff", True)  # sets BDF as ODE solver scheme
integ.setValue("relative_tolerance", 1e-6)
integ.setValue("absolute_tolerance", 1e-6)
integ.setValue("initial_time_step", 1e-6)

## 5. The statistical model

### Interindividual variability

We assume that there are **6 structural parameters** which vary between patients: 

  * `CLint`: intrinsic hepatic clearance
  * `KbBR`, `KbMU`, `KbAD`, `KbBO`, `KbRB`: tissue-to-blood partition coefficients for brain, muscle, adipose, bone, and rest of body

For participant $i$, let $\phi_i$ contain the 6 individual parameters. We assume they are log-normally distributed:

$$
\log \phi_i = \beta + \eta_i, \qquad \eta_i \sim \mathcal{N}(0,\Omega),
$$

where $\beta$ contains population means on the log-parameter scale, `eta_i` are the random effects and $\Omega$, the covariance matrix, describes interindividual variability.

Denoted by `"pdu"` (stands for Patient Descriptor Unknown) below. The user supplies:

  * `prior`: initial typical value **on the physical scale**: for example, `np.exp(7.6)` initializes the log-scale mean of `CLint` at 7.6
  * `prior_omega`: initial **log-scale variance**, so `CLint` starts with variance 4 and standard deviation 2. These are used to inform the first guess for covariance matrix $\Omega$

### Known effects

Denoted by `"pdk"` (stands for Patient Descriptor Known) below, these affect the structural model and are directly informed from the input dataset:

  * `WT`: weight 
  * `Dose`: the IV dose in mg

### Error model

The observations are assumed to be realizations of the deterministic model $f$ plus a zero-mean, $\sigma^2$-variance additive noise:

$$
y_{ij}=\log C_{ij}=f(\phi_i,\mathrm{Dose}_i,\mathrm{WT}_i,t_{ij})+\sigma\varepsilon_{ij},
\qquad \varepsilon_{ij}\sim\mathcal{N}(0,1).
$$

Initialized with variance 1 (`"initial_variance"` below)

See the [NLME model notes](../../docs/nlme_model.md) for the general formulation.

In [ ]:
statistical_model = {
    # "pdu" are parameters with interindividual variability
    "pdu": {
        "KbBR": {"prior": np.exp(1.1), "prior_omega": 0.5},
        "KbMU": {"prior": np.exp(0.3), "prior_omega": 0.5},
        "KbAD": {"prior": np.exp(2), "prior_omega": 0.5},
        "CLint": {"prior": np.exp(7.6), "prior_omega": 4},
        "KbBO": {"prior": np.exp(0.03), "prior_omega": 0.5},
        "KbRB": {"prior": np.exp(0.3), "prior_omega": 0.5},
    },
    # "pdk" are the KNOWN EFFECTS
    "pdk": {"WT", "Dose"},
    # The error model is assumed to be additive on the log-concentration, equivalent to multiplicative on the concentration
    "error_model": {
        "logC15": {"error_type": "additive", "initial_variance": 1},
    },
}

## 6. Configure SAEM

The goal is to maximise the observed-data likelihood of the population parameters $\Theta=(\beta,\Omega,\sigma^2)$, which requires integrating out each participant's unknown $\eta_i$. This integral has no closed form for an ODE model. SAEM avoids computing it by repeating three steps:

1. **Simulate** each participant's $\eta_i$ given their data and the current $\Theta$, using a few Metropolis–Hastings steps.
2. **Stochastic approximation**: update running summary statistics, $s_k = s_{k-1} + \gamma_k (S_k - s_{k-1})$.
3. **Maximize**: update $\Theta$ in closed form from $s_k$.

The step size $\gamma_k$ sets the phases: $\gamma_k = 1$ lets $\Theta$ move quickly, and a decreasing $\gamma_k$ averages out the simulation noise so the estimates converge.

The SAEM iterations happen in three consecutive phases:

| Phase | Iterations | What changes? |
| --- | --- | --- |
| Burn-in | 10 | MCMC states evolve while population parameters remain fixed |
| Learning | 90 | Population parameters update with $\gamma_k=1$, allowing rapid movement |
| Smoothing | 100 | Updates use $\gamma_m=m^{-0.8}$, where $m$ starts at 1 in this phase, to reduce stochastic fluctuations |

### Other settings

  * `nb_chains=1` uses one MCMC chain per participant during SAEM
  * `learning_rate_power=0.8` is `\gamma_k` update exponent during the smoothing phase (must be in the `]0.5, 1]` range)
  * `plot_frames=5` refreshes the fitting plots every five iterations. 

The final line below plots the **initial** parameter distributions. Check the starting distributions span plausible values.

See the [SAEM notes](../../docs/saem_implementation.md) for more information.

In [ ]:
config = Config(
    saem=SaemConfigDict(
        nb_iter_burnin=10,
        nb_iter_learning=90,
        nb_iter_smoothing=50,
        learning_rate_power=0.8,
        plot_frames=5,
    ),
    nlme=NlmeConfigDict(nb_chains=1),
    seed=YOUR_SEED,
)
nlme_model = NlmeModel(
    df=obs_df,
    input_params=statistical_model,
    structural_model=struct_model,
    config=config,
)
nlme_model.initial_estimates.plot_distribution()

## 7. Run the fit and read the traces

Watch the traces across the three phases: 

  * population parameters stay fixed during burn-in
  * move quickly during learning
  * settle during smoothing. 

A flat trace is necessary but not sufficient: it does not prove identifiability or a global maximum.

In [ ]:
nlme_model.optimizer.run()

## 8. Check population predictions

All following diagnostics hold the fitted population parameters fixed. Population diagnostics draw fresh random effects from $\mathcal{N}(0,\hat\Omega)$.

### Population weighted residuals (PWRES)

For participant $i$, simulated predictions estimate a mean $\mu_i$ and a predictive covariance $V_i$ that includes interindividual variability and residual error. With $L_iL_i^T=V_i$,

$$
\mathrm{PWRES}_i=L_i^{-1}(y_i-\mu_i).
$$

This centers, scales and decorrelates discrepancies. 
Look for a center near zero, a spread near one, and no systematic pattern over time. Normality is not guaranteed for nonlinear models. A persistent trend can point to a mismatch in the structural model, the error model, or both.

In [ ]:
nlme_model.diagnostics.compute_population_diagnostics(nb_samples=100)
nlme_model.plot.weighted_residuals("pwres", time_unit="hour")

### Normalized prediction distribution errors (NPDE)

NPDE ranks observed PWRES against noisy simulations decorrelated with the same $\mu_i$ and $L_i$
then maps the quantiles to a standard-normal score.
Under an adequate model, the reference is approximately $\mathcal{N}(0,1)$: a displaced center suggests bias, while excess spread or unusual tails suggest a variability mismatch.

Note that the empirical percentiles are clipped to keep the scores finite, so few replicates will provide limited resolution in the tails.

In [ ]:
nlme_model.plot.weighted_residuals("npde", time_unit="hour")

### Visual predictive check (VPC)

The VPC compares observed concentration quantiles over time with quantiles from simulations using the population distribution. 
Each replicate preserves the observed study design and observation times, draws fresh population random effects, and adds residual noise.

With the current implementation defaults, it uses **10 time bins** and the **5th, 50th, and 95th percentiles**. 
Each shaded band spans the 5th to 95th percentile of a simulated sample quantile: a central **90% interval** across replicates.

This plot pools the dose groups so stratification by dose would be a useful follow-up.

In [ ]:
nlme_model.plot.vpc(time_unit="hour")

## 9. Explore individual estimates and parameter information

### Empirical Bayes estimates (EBEs) and approximate MAP fits

Now condition on each participant's own observations:

$$
p(\eta_i\mid y_i;\hat\Theta)\propto
p(y_i\mid\eta_i;\hat\Theta)\,p(\eta_i;\hat\Omega).
$$

We run `nb_samples` conditional sampling steps with population parameters fixed. Individual estimates balance agreement with the participant's data against plausibility under the fitted population distribution; this is the empirical Bayes step.

Here, the reported **MAP** (maximum a posteriori) estimate is the highest conditional-density state visited for each participant.

The first plot overlays individual fitted profiles and observations.  
The second plot compares observed and fitted values with the identity line.

In [ ]:
nlme_model.diagnostics.sample_conditional_distribution(nb_samples=100)
nlme_model.plot.map_estimates(time_unit="hour")
nlme_model.plot.map_estimates_gof()

### Individual weighted residuals (IWRES)

IWRES standardize the discrepancy from each participant's approximate MAP prediction. 
Here the residual model is additive on log concentration, so we have:

$$
\mathrm{IWRES}_{ij}=\frac{y_{ij}-f(\hat\phi_i,\mathrm{Dose}_i,\mathrm{WT}_i,t_{ij})}{\hat\sigma}.
$$

Look for residuals centered near zero and no persistent time trend. The standard-normal curve is only a rough visual reference.

In [ ]:
nlme_model.plot.weighted_residuals("iwres", time_unit="hour")

### Conditional parameter distributions and relationships

The next plots compare retained conditional parameter samples with approximate MAP estimates, using marginal histograms and pairwise scatterplots on logarithmic axes. 
Strong pairwise relationships can suggest compensating parameter combinations. They are clues for further investigation, not proof of a biological correlation.

In [ ]:
nlme_model.plot.conditional_codistributions()

### Quantify shrinkage

The table reports standard-deviation-based shrinkage of the approximate MAP random effects across participants:

$$
\mathrm{shrinkage}_k=100\left(1-
\frac{\mathrm{SD}_i(\hat\eta_{ik}^{\mathrm{MAP}})}{\sqrt{\hat\Omega_{kk}}}
\right)\%.
$$

A large positive value means the individual estimates are less dispersed than the fitted population distribution, often because the individual observations contain limited information about that parameter. 
In such a case, the EBE and IWRES are unreliable [5].


This affects how much weight to put on EBE-based covariate plots or parameter correlations. Near-zero shrinkage indicates similar spreads; finite-sample estimates can also be negative.


In [ ]:
nlme_model.diagnostics.compute_shrinkage(nb_samples=100)
parameter_names = nlme_model.diagnostics.model.pdu_names
shrinkage_percent = nlme_model.diagnostics.shrinkage.detach().cpu().numpy() * 100
shrinkage_df = pd.DataFrame(
    [shrinkage_percent],
    columns=parameter_names,
    index=["shrinkage(%)"],
)

display(shrinkage_df.round(1))

### Estimate the observed-data log-likelihood

This is actually a population-level quantity but we need to have conditional samples to compute it, that is why it only appears now.

After fitting, importance sampling approximates the integral that defines the observed-data log-likelihood. The implementation builds a Student-$t$ proposal $q_i$ from the retained conditional samples and draws 100 proposal samples per participant. Each draw is weighted by its joint observation-and-random-effect density divided by its proposal density:

$$
\widehat\ell(\hat\Theta;y)=\sum_i\log\left[
\frac{1}{M}\sum_{m=1}^{M}
\frac{p(y_i\mid\eta_i^{(m)};\hat\Theta)\,p(\eta_i^{(m)};\hat\Omega)}
{q_i(\eta_i^{(m)})}\right],\qquad \eta_i^{(m)}\sim q_i.
$$

Larger values indicate better likelihood fit when comparing the same observations, transformation, and likelihood conventions. Model comparison also needs to account for model complexity (using BIC or AIC for instance).

In [ ]:
nlme_model.diagnostics.compute_log_likelihood_importance_sampling(
    nb_proposal_samples=100
)
print(
    f"Observed-data log-likelihood estimate, -2LL = {-2*nlme_model.diagnostics.importance_sampler.log_lik:.2f}"
)

## 10. Follow-up simulations in Jinkō — optional extension

The fitted individual parameter sets can now serve as a virtual population for mechanistic simulations. This section requires the Jinkō Python client, configured credentials, and access to the destination folder and matching computational model. The cells upload a virtual population and create and run a trial in that workspace.

### Connect to the Jinkō workspace

Import the client, then initialize it using the credentials configured in the execution environment. `auth_check()` checks the connection before the upload and trial-creation steps.

In [ ]:
from jinko import JinkoClient

In [ ]:
client = JinkoClient()
client.auth_check()

### Select the destination and computational model

`FOLDER_ID` selects the destination for the created objects. 
Create your own folder and replace the placeholder below with its ID (the team can help your for that).

`MODEL_SID` identifies the Jinkō model to simulate and `DATA_TABLE_SID` the aggregated observations. They're the same for everybody so they don't need changing.

In [ ]:
FOLDER_ID = "5a55a90b-5628-4acd-bcfe-f1ab8ce35dd5"
MODEL_SID = "cm-F0YC-RF9m"
DATA_TABLE_SID = "dt-RS4A-OiI7"

### Upload the virtual population

This cell creates a Jinkō virtual population from the prepared parameter table and prints its link. The upload contains individual model inputs; the fitted observation-error model is not part of this table.

In [ ]:
conditional_samples = nlme_model.diagnostics.sampler.total_samples_parameters_df
vpop_df = conditional_samples.rename(columns={"id": "patientIndex"}).drop(
    columns=["id_ref"]
)

vpop = client.create_vpop_from_dataframe(
    df=vpop_df,
    name="SAEM vpop",
    description="SAEM conditional samples",
    folder=FOLDER_ID,
)
print(f"Vpop link: {vpop.url}")

### Define the simulation trial

Load the selected computational model, create an output set requesting `logC15`, and create a trial using the uploaded virtual population. The printed trial link opens the resulting configuration in Jinkō.

In [ ]:
model = client.get_model(MODEL_SID)
data_table = client.get_data_table(DATA_TABLE_SID)
trial_output_timeseries = model.create_simple_output_set(
    measures=["logC15"],
    name="Timeseries to output",
    folder=FOLDER_ID or None,
)
trial = model.create_trial(
    vpop=vpop,
    data_tables=[
        {
            "dataTableId": {
                "coreItemId": data_table.core_id,
                "snapshotId": data_table.snapshot_id,
            },
            "options": {},
            "include": True,
        }
    ],
    simple_output_set=trial_output_timeseries,
    name="Mavoglurant trial",
    folder=FOLDER_ID,
)
print(f"Trial link: {trial.url}")

### Run the trial

Submit the trial and wait for completion before creating its visualization. Each virtual participant corresponds to a retained individual parameter set, so the simulation workload can be much larger than the original 120-person cohort.

In [ ]:
trial.run()
trial.wait_until_completed()

### Open a visualization workspace

The final execution cell creates an **trial visualization** and prints its link. Open it in Jinkō to configure the plots for the simulated outputs.

In [ ]:
visualization = trial.create_empty_trial_visualization(
    name="Trial visualization", folder=FOLDER_ID
)
visualization.data_overlay.add_table(
    data_table, label="Aggregated_Observed", include=True
)
print(f"Trial visualization link: {visualization.url}")

## References

[1] Delyon, Lavielle & Moulines (1999), [*Convergence of a stochastic approximation version of the EM algorithm*](https://doi.org/10.1214/aos/1018031103)

[2] Kuhn & Lavielle (2005), [*Maximum likelihood estimation in nonlinear mixed effects models*](https://doi.org/10.1016/j.csda.2004.07.002)

[3] The Mavoglurant dataset, [*nlmixr2 R vignette*](https://nlmixr2.org/articles/mavoglurant.html)

[4] Wendling _et al._ (2016), [*Reduction of a whole-body physiologically based pharmacokinetic model to stabilise the Bayesian analysis of clinical data*](https://doi.org/10.1208/s12248-015-9840-7)

[5] Savic & Karlsson (2009), [*Importance of Shrinkage in Empirical Bayes Estimates for Diagnostics: Problems and Solutions*](https://link.springer.com/article/10.1208/s12248-009-9133-0)
